# 1순위 고성능 GPU 모델 서버 (Kaggle Notebooks 용)

1. 오른쪽 설정(Settings)에서 **Accelerator → GPU T4 x2 (또는 GPU P100)** 와 **Internet → On** 을 켭니다.
2. 아래 코드 셀을 실행하면 `Ollama` + 상위 모델(`gemma3:12b`, `qwen3:8b`) + 외부 연결 주소(`https://....trycloudflare.com`)가 자동으로 출력됩니다.
3. 출력된 `https://....trycloudflare.com` 주소를 Streamlit Cloud의 `Secrets` (`OLLAMA_PRIMARY_URL` 또는 `OLLAMA_BASE_URL`)에 붙여넣으세요.
4. 내 컴퓨터를 끄고 싶을 때는 오른쪽 위 **`Save Version` → `Save & Run All (Commit)`** 을 누르면 캐글 서버에서 최대 9~12시간 동안 혼자 돌아갑니다.

## 실험 노트북과의 차이
이 파일은 웹앱 연결용 임시 모델 서버를 시작하는 예제입니다. 모델 비교 결과를 생성하는 파일은 `kaggle_model_comparison.ipynb`입니다.
두 GPU 노트북을 동시에 시작하지 말고 계정의 동시 세션 한도를 확인하세요.
외부 URL은 세션 재시작 시 바뀔 수 있습니다. 기본 터널 예제 자체에 사용자 인증은 없으므로 주소 공개를 주의하고, 실제 서비스에는 인증 프록시가 필요합니다.
세션 유지 셀의 시간 경과 출력은 서버 상태 검사 결과가 아닙니다. 백그라운드 실행에도 캐글의 세션·주간 한도가 적용됩니다.


In [ ]:
import re
import subprocess
import time

print('1) zstd, Ollama 및 Cloudflared 설치 중...')
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run(
    'curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
    shell=True,
    check=True,
)

print('2) Ollama GPU 서버 백그라운드 시작...')
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

print('3) 1순위 상위 모델 다운로드 중 (gemma3:12b, qwen3:8b)...')
subprocess.run(['ollama', 'pull', 'gemma3:12b'], check=True)
subprocess.run(['ollama', 'pull', 'qwen3:8b'], check=True)

print('4) 외부 접속용 Cloudflare 터널 시작...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434', '--http-host-header', 'localhost:11434', '--no-autoupdate'],
    stdout=open('/tmp/cloudflared.log', 'w'),
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log_text = open('/tmp/cloudflared.log', 'r', errors='ignore').read()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
    if m:
        public_url = m.group(0)
        break

print('=' * 72)
print('캐글 GPU 1순위 서버 준비 완료!')
print(f'OLLAMA_PRIMARY_URL = "{public_url}"')
print('=' * 72)

In [ ]:
# 백그라운드 실행(Save & Run All) 시 세션이 바로 종료되지 않도록 최대 11시간 동안 서버를 유지합니다.
import time

print('서버 유지 대기 중... (내 컴퓨터를 꺼도 캐글에서 계속 동작합니다)')
for minute in range(660):
    time.sleep(60)
    if minute % 30 == 0:
        print(f'{minute}분 경과 - GPU Ollama 서버 정상 실행 중')